In [ ]:
// load the formatting setup 
#load "../setup.fsx"

# <a href="https://en.wikipedia.org/wiki/Legendre%27s_conjecture">Legendre's Conjecture</a>

The function `divisors` takes one argument:
- `k` is a positive natural number.

The function returns the set of all natural numbers `t` from the range $\{1, \cdots, k\}$ that divide `k` evenly, i.e. the set of all `t` such that `k % t = 0`.

The expression `[ for t in 1 .. k do if k % t = 0 then t ]` is a *list comprehension*: it iterates over all numbers `t` from `1` to `k` and collects those numbers that divide `k`.  The function `set` converts the resulting list into a set.

In [ ]:
let divisors (k: int) : Set<int> =
    set [ for t in 1 .. k do if k % t = 0 then t ]

In [ ]:
divisors 24

The function `isPrime` takes one argument:
- `k` is a positive natural number.

The function returns `true` if `k` is a prime number and `false` otherwise.  A number $k$ is prime if and only if $k \not= 1$ and the only divisors of $k$ are $1$ and $k$, i.e. the set of divisors of $k$ is the set $\{1, k\}$.

In [ ]:
let isPrime (k: int) : bool =
    k <> 1 && divisors k = set [1; k]

More than 200 years ago the French mathematician
<a href="http://de.wikipedia.org/wiki/Adrien-Marie_Legendre">Adrien-Marie Legendre</a>
(1752 - 1833) [conjectured](https://en.wikipedia.org/wiki/Legendre%27s_conjecture)  that there is always a *prime number* between two successive squares bigger than one, i.e. if we denote the set of all *prime numbers* as $\mathbb{P}$, then we have:
$$ \forall n \in \mathbb{N}: \bigl(n \geq 1 \rightarrow \exists p \in \mathbb{P}: n^2 < p < (n+1)^2 \bigr) $$
The question whether this claim is true is still **open**.

The function `legendre` takes one argument:
- `n` is a natural number such that $n \geq 1$.

The function checks whether there is a prime number $p$ such that
$$ n^2 < p < (n+1)^2 $$
holds.  The function returns `true` if there is a prime number $p$ between $n^2$ and $(n+1)^2$.  In this case, it also prints this prime number.
Otherwise, `false` is returned.

**Implementation:** The expression `seq { n*n + 1 .. (n+1)*(n+1) - 1 }` denotes the sequence of all numbers $k$ such that $n^2 < k < (n+1)^2$.  This sequence is *lazy*: its elements are only computed when they are needed.  The function `Seq.tryFind isPrime` searches this sequence for the first prime number.  It returns
- `Some p` if `p` is the first prime number in the sequence, and
- `None` if the sequence does not contain a prime number.

As the sequence is lazy, the search stops as soon as the first prime number has been found.

In [ ]:
let legendre (n: int) : bool =
    match Seq.tryFind isPrime (seq { n*n + 1 .. (n+1)*(n+1) - 1 }) with
    | Some p -> printfn "%d^2 < %d < %d^2" n p (n+1)
                true
    | None   -> false

Let us test the functions `divisors` and `isPrime`.  The function `print` is defined in the file `setup.fsx` that is loaded in the first cell of this notebook.  It prints a value using F# syntax.

In [ ]:
for i in 1 .. 9 do
    print (divisors i)

In [ ]:
set [ for p in 2 .. 99 do if isPrime p then p ]

We are ready to test the function `legendre`:

In [ ]:
legendre 2

In [ ]:
legendre 3

In [ ]:
legendre 4

This is getting tedious, lets automate the testing.

The function `findCounterExample` takes one argument:
- `n` is the natural number where the search for a counter example starts.

The function tries to find a counter example for Legendre's conjecture, i.e. it tries to find a natural number $m \geq n$ such that there is no prime number $p$ between $m^2$ and $(m+1)^2$.  Once a counter example $m$ is found, the function stops and prints the natural number $m$.  Otherwise, the function keeps going until the sun rises in the west.  The function does not return a meaningful result: its return type is `unit`.

**Implementation:** F# has no `while True` loop with a `return` statement.  Instead, the loop is implemented as a recursive function: if `legendre n` is `true`, the function calls itself with the argument `n + 1`.  As this recursive call is the last action of the function (a so called *tail call*), the F# compiler translates the recursion into a loop.  Hence, the recursion does not consume memory on the stack.

**Remark:** As `n` has the type `int`, i.e. it is a 32-bit integer, the computation of `n*n` overflows if `n` is bigger than $46\,340$.  In practice, this is not an issue, since the function would need a very long time to get that far.

In [ ]:
let rec findCounterExample (n: int) : unit =
    if legendre n then
        findCounterExample (n + 1)
    else
        printfn "Counter example found: No prime between %d^2 and %d^2!" n (n+1)

In our quest to get rich and famous, we start the function and wait for a counter example...

**Warning:** The next cell does not terminate.  You have to interrupt the kernel to stop it.

In [ ]:
findCounterExample 1

The function call `findCounterExample 1` will stop if and only if Legendre's conjecture is false.  Therefore, if we had a function that could take our definition of the function `findCounterExample` and a natural number `n` as inputs and that could then decide, whether the function call `findCounterExample n` would eventually terminate, then this function would be able to decide whether Legendre's conjecture is correct.